In [1]:
# 데이터 불러오기

from torchvision import datasets
from torchvision.transforms import ToTensor

In [2]:
# 학습 데이터 불러오기
train_data = datasets.FashionMNIST(
            root="../Data" , # 어디에 저장할 건지
            train=True, # train 인지 
            download=True,# 다운로드 해야 함
            transform=ToTensor() # 이렇게 바꿔줌
)

In [3]:
# 테스트 데이터 불러오기
test_data = datasets.FashionMNIST(
            root="../Data" , # 어디에 저장할 건지
            train=False,
            download=True,
            transform=ToTensor()
)

In [4]:
# 받은 데이터 확인
print(train_data.data.shape)
print(train_data.targets.shape)
print(test_data.data.shape)
print(test_data.targets.shape)

torch.Size([60000, 28, 28])
torch.Size([60000])
torch.Size([10000, 28, 28])
torch.Size([10000])


In [5]:
# data와 target 분류
train_input = train_data.data
train_target = train_data.targets

test_input = test_data.data
test_target = test_data.targets
# 간단하게 쓰려고 변수 정해준 거 

In [6]:
# target 확인 
train_target[:10]

tensor([9, 0, 0, 3, 0, 2, 7, 2, 5, 5])

In [7]:
import numpy as np

print(np.unique(train_target, return_counts=True))

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), array([6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000]))


In [8]:
# 데이터 표준화 및 2차원 행렬
train_scaled = (train_input / 255.0).reshape(-1, 28*28)
test_scaled = (test_input / 255.0).reshape(-1, 28*28)

print(train_scaled.shape)
print(test_scaled.shape)

torch.Size([60000, 784])
torch.Size([10000, 784])


----
#### 모델 만들기

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [10]:
# Train과 Valid
from sklearn.model_selection import train_test_split

train_scaled, val_scaled, train_target, val_target = train_test_split(
                                                        train_scaled,
                                                        train_target,
                                                        test_size=0.2,
                                                        random_state=42
)

In [11]:
# Dataset과 Dataloader 생성

batch_size = 32 # mini batch
train_dataset = TensorDataset(train_scaled, train_target)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True) # 한번 epoch가 발생했을때 섞어 쓰는 거

val_dataset = TensorDataset(val_scaled, val_target)
val_loader = DataLoader(val_dataset, batch_size=batch_size) # 벨리드나 테스트는 검증이라 셔플을 쓰지 않는다 

#### 모델 정의

In [12]:
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__() # super에 있는 모델을 쓰겠다는 거 
        self.dense = nn.Linear(28*28, 10) # 입력층
        self.softmax = nn.Softmax(dim=1) # 앞에는 변수임
    
    def forward(self, x) : 
        x = self.dense(x) # 들어온 데이터로 층 만든것
        return self.softmax(x)

In [13]:
# 모델 인스턴스
model = SimpleModel()

In [14]:
# 손실함수와 옵티마이져
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters()) # 모델에 파라메터를 가지고 쓰겠다 -> 모델의 가중치값을 가지고 변환 시키겠다 

---
### 모델 훈련

In [15]:
# 학습 함수
def train(model, train_loader, criterion, optimizer, device):
    model.train()
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device) # 디바이스 (cpu)로 보냄
        optimizer.zero_grad() # 초기화 시켜주는 거 _옵티마이저가 곱하기 하는거라서 
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    return loss.item()

In [ ]:
# 평가함수 
def evaluate(model,val_loader,criterion,device):
    model.eval()
    total_loss = 0 # 전체 손실 합계 
    correct = 0  # 정확하게 예측한 샘플 수 
    total = 0 # 전체 샘플 수
    with torch.no_grad():
        for inputs ,targets in val_loader: # 문제 정답 넣기
            inputs,targets = inputs.to(device),targets.to(device) # 문제 정답 디바이스로 보내기
            outputs = model(inputs)
            loss = criterion(outputs,targets)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return total_loss / len(val_loader), correct / total

---
### 학습 및 평가

In [17]:
device = torch.device("mps" if torch.backends.mps.is_available() else 'cpu')
print(device)
# model.to(device)

mps


In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
print(device)
model.to(device)

cpu


SimpleModel(
  (dense): Linear(in_features=784, out_features=10, bias=True)
  (softmax): Softmax(dim=1)
)

In [19]:
num_epochs = 100

for epoch in range(num_epochs):
    train_loss = train(model,train_loader,criterion,optimizer,device)
    print(f'Epoch[{epoch+1:>3}/ {num_epochs}],Loss : {train_loss:.4f}')

Epoch[  1/ 100],Loss : 1.7045
Epoch[  2/ 100],Loss : 1.7314
Epoch[  3/ 100],Loss : 1.5896
Epoch[  4/ 100],Loss : 1.6675
Epoch[  5/ 100],Loss : 1.5425
Epoch[  6/ 100],Loss : 1.6641
Epoch[  7/ 100],Loss : 1.6860
Epoch[  8/ 100],Loss : 1.7059
Epoch[  9/ 100],Loss : 1.7726
Epoch[ 10/ 100],Loss : 1.5677
Epoch[ 11/ 100],Loss : 1.6488
Epoch[ 12/ 100],Loss : 1.5877
Epoch[ 13/ 100],Loss : 1.7915
Epoch[ 14/ 100],Loss : 1.6474
Epoch[ 15/ 100],Loss : 1.6752
Epoch[ 16/ 100],Loss : 1.7722
Epoch[ 17/ 100],Loss : 1.7103
Epoch[ 18/ 100],Loss : 1.6675
Epoch[ 19/ 100],Loss : 1.6682
Epoch[ 20/ 100],Loss : 1.5371
Epoch[ 21/ 100],Loss : 1.5310
Epoch[ 22/ 100],Loss : 1.5768
Epoch[ 23/ 100],Loss : 1.5910
Epoch[ 24/ 100],Loss : 1.6043
Epoch[ 25/ 100],Loss : 1.7281
Epoch[ 26/ 100],Loss : 1.7405
Epoch[ 27/ 100],Loss : 1.6977
Epoch[ 28/ 100],Loss : 1.6666
Epoch[ 29/ 100],Loss : 1.5529
Epoch[ 30/ 100],Loss : 1.7593
Epoch[ 31/ 100],Loss : 1.6096
Epoch[ 32/ 100],Loss : 1.6767
Epoch[ 33/ 100],Loss : 1.6309
Epoch[ 34/

In [21]:
# 훈련 평가
train_loss, train_accuarcy = evaluate(model, train_loader, criterion, device)
print(f"Loss : {train_loss}, Accuracy : {train_accuarcy}")

Loss : 1.5743997552394866, Accuracy : 0.893125


In [22]:
#검증 평가
val_loss, val_accuarcy = evaluate(model, val_loader, criterion, device)
print(f"Loss : {val_loss}, Accuracy : {val_accuarcy}")

Loss : 1.6028659114837647, Accuracy : 0.8590833333333333


In [ ]:
# 일반화 평가
test_dataset = TensorDataset(test_scaled, test_target)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


# 평가하기
test_loss, test_accuarcy = evaluate(model, test_loader, criterion, device)
print(f"Loss : {test_loss}, Accuracy : {test_accuarcy}")

Loss : 1.6122635839084467, Accuracy : 0.8491
